In [1]:
from dotenv import load_dotenv

# Load OPENAI_API_KEY (and any other secrets) from a local .env file.
load_dotenv()


True

# Notebook 4 · Team (Modular) Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Use Case

---

## Architecture Overview

In a **team MAS** each agent owns exactly one non-overlapping section of a
shared project board. Work flows as a relay: each specialist fills their
section and passes the enriched board to the next.

```
USER REQUEST
     │
     ▼
┌──────────────┐   ┌──────────────┐   ┌──────────────┐
│  Destination │──►│    Flight    │──►│    Hotel     │
│   Designer   │   │  Specialist  │   │  Specialist  │
└──────────────┘   └──────────────┘   └──────────────┘
                                             │
                                             ▼
                                   ┌──────────────────┐
                                   │Activity Specialist│
                                   └────────┬──────────┘
                                            │
                                            ▼
                                   ┌──────────────────┐
                                   │ Itinerary Writer  │
                                   └──────────────────┘
                each role owns one state field · relay pattern
```

## Pattern in this notebook

| LangChain concept | Role in team MAS |
|---|---|
| `AgentState` | One field per board section (5 fields total) |
| `create_agent` (specialists) | Each owns one section, reads but does not modify others |
| `@tool` + `ToolRuntime` | Reads full board from state, writes own field only |
| `Command(update={"field": ...})` | Writes exactly one field per tool call |
| `create_agent` (coordinator) | Calls specialists in fixed relay order |


## 1 · Setup

In [2]:
# ── Traveler request ─────────────────────────────────────────────────────────
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use options listed here.
DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",       "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",       "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",        "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",        "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",    "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel", "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
def flights_for(destination: str)    -> list: return [f for f in FLIGHTS    if f["destination"] == destination]
def hotels_for(destination: str)     -> list: return [h for h in HOTELS     if h["destination"] == destination]
def activities_for(destination: str) -> list: return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Custom State

`TeamState` extends `AgentState` with **one field per board section**.
Each tool writes to exactly one field — section ownership is encoded in the
state schema itself.

```
TeamState
  ├─ destination_and_period  ← destination_designer writes here
  ├─ flight                  ← flight_specialist writes here
  ├─ hotel                   ← hotel_specialist writes here
  ├─ activities              ← activity_specialist writes here
  └─ final_writeup           ← itinerary_writer writes here
```


In [3]:
from langchain.agents import AgentState

class TeamState(AgentState):
    # Each field is owned by exactly one specialist.
    # Fields start empty and are filled in relay order.
    destination_and_period: str  # → destination_designer
    flight:                  str  # → flight_specialist
    hotel:                   str  # → hotel_specialist
    activities:              str  # → activity_specialist
    final_writeup:           str  # → itinerary_writer


## 3 · Specialist Sub-Agents

Five specialists, each with a tightly scoped system prompt.
The prompt explicitly forbids writing outside their owned section — this
enforces the non-overlapping ownership contract in the LLM's behaviour.


In [4]:
from langchain.agents import create_agent

destination_designer = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Destination Designer in a modular travel-agency team.\n"
        "You own exactly one section of the project board: 'destination_and_period'.\n"
        "Read the full board, then write your section ONLY — "
        "do not mention flights, hotels, or activities.\n"
        "Only use destinations from the catalog."
    ),
)

flight_specialist = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Flight Specialist in a modular travel-agency team.\n"
        "You own exactly one section: 'flight'.\n"
        "The destination has already been chosen (see board). "
        "Choose ONE flight for that destination. Write the 'flight' section ONLY.\n"
        "Only use flights from the catalog."
    ),
)

hotel_specialist = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Hotel Specialist in a modular travel-agency team.\n"
        "You own exactly one section: 'hotel'.\n"
        "The destination has already been chosen (see board). "
        "Choose ONE hotel for that destination. Write the 'hotel' section ONLY.\n"
        "Only use hotels from the catalog."
    ),
)

activity_specialist = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Activity Specialist in a modular travel-agency team.\n"
        "You own exactly one section: 'activities'.\n"
        "The destination has already been chosen (see board). "
        "Choose 2–3 activities (food + culture mix). Write the 'activities' section ONLY.\n"
        "Only use activities from the catalog."
    ),
)

itinerary_writer = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are the Itinerary Writer in a modular travel-agency team.\n"
        "You own exactly one section: 'final_writeup'.\n"
        "All other sections are complete (see board). "
        "Turn them into one clean, client-ready 4-day itinerary. "
        "Write the 'final_writeup' section ONLY — do not change any earlier decisions."
    ),
)

print("Specialist sub-agents created.")


Specialist sub-agents created.


## 4 · Specialist Tools (One Field per Tool)

Each tool follows the relay pattern:

1. **Read full board** from state (all fields, including earlier sections).
2. **Invoke specialist** — they see the full board but must write only their field.
3. **Write one field** via `Command(update={"<field>": result, ...})`.

> **Team property:** `Command(update=...)` only names the one field this
> specialist owns. All other fields are untouched.


In [5]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command


def _board_text(state: dict) -> str:
    """Render the current board state as readable text for specialist prompts."""
    fields = ["destination_and_period", "flight", "hotel", "activities", "final_writeup"]
    lines = []
    for f in fields:
        val = state.get(f, "") or "(not filled yet)"
        lines.append(f"  {f}: {val}")
    return "\n".join(lines)


@tool
async def run_destination_designer(runtime: ToolRuntime) -> str:
    '''Destination Designer fills the 'destination_and_period' section.'''
    response = await destination_designer.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Project board (current state):\n{_board_text(runtime.state)}\n\n"
            "Write the 'destination_and_period' section."
        ))]
    })
    result = response["messages"][-1].content
    return Command(update={
        "destination_and_period": result,
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def run_flight_specialist(runtime: ToolRuntime) -> str:
    '''Flight Specialist fills the 'flight' section.'''
    response = await flight_specialist.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Project board (current state):\n{_board_text(runtime.state)}\n\n"
            "Write the 'flight' section."
        ))]
    })
    result = response["messages"][-1].content
    return Command(update={
        "flight": result,
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def run_hotel_specialist(runtime: ToolRuntime) -> str:
    '''Hotel Specialist fills the 'hotel' section.'''
    response = await hotel_specialist.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Project board (current state):\n{_board_text(runtime.state)}\n\n"
            "Write the 'hotel' section."
        ))]
    })
    result = response["messages"][-1].content
    return Command(update={
        "hotel": result,
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def run_activity_specialist(runtime: ToolRuntime) -> str:
    '''Activity Specialist fills the 'activities' section.'''
    response = await activity_specialist.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Project board (current state):\n{_board_text(runtime.state)}\n\n"
            "Write the 'activities' section."
        ))]
    })
    result = response["messages"][-1].content
    return Command(update={
        "activities": result,
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def run_itinerary_writer(runtime: ToolRuntime) -> str:
    '''Itinerary Writer fills the 'final_writeup' section from the completed board.'''
    response = await itinerary_writer.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Project board (all sections filled):\n{_board_text(runtime.state)}\n\n"
            "Write the 'final_writeup' section."
        ))]
    })
    result = response["messages"][-1].content
    return Command(update={
        "final_writeup": result,
        "messages": [ToolMessage(result, tool_call_id=runtime.tool_call_id)],
    })

print("Specialist tools defined.")


Specialist tools defined.


## 5 · Pipeline Coordinator

The coordinator's only job is to call the five specialists in the correct
dependency order. The system prompt encodes this relay as a numbered list.

> No manager logic here — the pipeline order is fixed because the task has
> a natural dependency structure (you must know the destination before you
> can choose a flight, hotel, or activities).


In [6]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[
        run_destination_designer,
        run_flight_specialist,
        run_hotel_specialist,
        run_activity_specialist,
        run_itinerary_writer,
    ],
    state_schema=TeamState,
    system_prompt=(
        "You are the pipeline coordinator of a modular travel-agency team.\n"
        "You do NOT make any planning decisions yourself.\n\n"
        "Call the specialists in this exact order — do not skip or reorder:\n"
        "1. run_destination_designer\n"
        "2. run_flight_specialist\n"
        "3. run_hotel_specialist\n"
        "4. run_activity_specialist\n"
        "5. run_itinerary_writer\n\n"
        "After all five have run, confirm that the project board is complete "
        "and present the final_writeup to the user."
    ),
)

print("Team pipeline coordinator created.")


Team pipeline coordinator created.


## 6 · Run

In [15]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content=USER_REQUEST)],
        # All sections start empty — filled in relay order
        "destination_and_period": "",
        "flight":                 "",
        "hotel":                  "",
        "activities":             "",
        "final_writeup":          "",
    }
)


In [16]:
from pprint import pprint
pprint(response)


{'activities': 'activities:\n'
               '  - Old Town walking tour | culture | EUR 18\n'
               '  - Czech dinner and jazz night | food | EUR 30',
 'destination_and_period': 'destination_and_period: Prague in spring '
                           '(April–June)',
 'final_writeup': 'final_writeup: |\n'
                  '  Enjoy a delightful 4-day spring getaway to Prague, '
                  'blending rich culture and delicious food with easy '
                  'logistics and comfortable accommodations. Flying direct '
                  'from Rome with a quick 1 hour and 55-minute flight on '
                  'FR-721, you’ll arrive refreshed and ready to explore.\n'
                  '\n'
                  '  Stay centrally at the charming Old Town House, a historic '
                  'hotel just steps from Prague’s main sights, perfectly '
                  'situated for your cultural explorations and evening '
                  'outings. Days are kept simple and balance

In [17]:
# Final itinerary written by the itinerary_writer specialist
print(response["messages"][-1].content)


Your 4-day spring trip from Rome is planned in Prague, a perfect destination for the season from April to June.

Flights: You will fly direct on airline FR-721, with a flight duration of about 1 hour 55 minutes, at a cost of around EUR 110.

Hotel: Accommodation is booked at the Old Town House, a historic hotel centrally located near Prague's main sights, with a rate of EUR 115 per night.

Activities: Your days will be a mix of culture and food, including an Old Town walking tour (€18) and a Czech dinner with jazz night (€30).

The plan offers a mid-range budget, easy flights, central lodging, and a simple daily schedule blending local culture and cuisine, ensuring a relaxed and enjoyable experience in Prague this spring.


In [18]:
# Inspect individual board sections from state
for field in ["destination_and_period", "flight", "hotel", "activities", "final_writeup"]:
    print(f"\n── {field} ──")
    print(response.get(field, "(empty)"))



── destination_and_period ──
destination_and_period: Prague in spring (April–June)

── flight ──
flight:
  airline: FR-721
  type: direct
  price: EUR 110
  duration: 1h 55m

── hotel ──
hotel: Old Town House | EUR 115/night | historic hotel near main sights

── activities ──
activities:
  - Old Town walking tour | culture | EUR 18
  - Czech dinner and jazz night | food | EUR 30

── final_writeup ──
final_writeup: |
  Enjoy a delightful 4-day spring getaway to Prague, blending rich culture and delicious food with easy logistics and comfortable accommodations. Flying direct from Rome with a quick 1 hour and 55-minute flight on FR-721, you’ll arrive refreshed and ready to explore.

  Stay centrally at the charming Old Town House, a historic hotel just steps from Prague’s main sights, perfectly situated for your cultural explorations and evening outings. Days are kept simple and balanced: start with an insightful Old Town walking tour to immerse yourself in Prague’s fascinating history a

## 7 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| `TeamState` with one field per section | Section ownership encoded directly in the schema |
| Each tool updates **only its own field** | Non-overlapping responsibilities enforced in code |
| Specialists read the full board via `_board_text(runtime.state)` | Later specialists benefit from all earlier decisions |
| Coordinator calls tools in fixed relay order | Dependencies are explicit, not managed dynamically |

---

## Series Summary

| | Flat | Hierarchical | Society | Team |
|---|---|---|---|---|
| State field | `shared_board` | `manager_notes` | `public_board` + `winning_package` | One field per section |
| Authority | None — peers decide | Manager decides | Social rules → majority vote | Section ownership |
| Tool `Command` update | Appends to board list | Appends to notes list | Appends ballot entry | Writes one string field |
| Coordinator role | Thin cycler | Planner + synthesiser | Vote counter | Fixed relay caller |
| Stops when | All peers vote ready | Manager done | Majority reached | Pipeline complete |
